Turn the video into frame

In [1]:
import cv2, os

In [9]:

cap = cv2.VideoCapture("sample.mp4")

if not cap.isOpened():
    raise RuntimeError("Could not open sample.mp4")

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)

print("FPS:", fps)
print("Total frames:", total_frames)
print("Duration (seconds):", total_frames / fps if fps > 0 else "Unknown")

FPS: 25.0
Total frames: 1341.0
Duration (seconds): 53.64


In [10]:
print("Current working directory:", os.getcwd())

Current working directory: c:\Users\jetch\OneDrive\Desktop\JJ\2026\Job Application\footfall_cam


In [11]:
os.makedirs("frames_25fps_every_5", exist_ok=True)

i = 0
saved = 0

while True:
    ok, frame = cap.read()

    if not ok:
        break

    if i % 5 == 0:
        filename = f"frames_25fps_every_5/{i:06d}.jpg"

        if cv2.imwrite(filename, frame):
            saved += 1
        else:
            print(f"Failed to save {filename}")

    i += 1


print(f"Read {i} frames and saved {saved} images.")

Read 1341 frames and saved 269 images.


In [6]:
cap.release()

Split the data downloaded back from CVAT

In [4]:
images_path = 'frames_25fps_every_5/images'
labels_path = 'frames_25fps_every_5/labels'

In [5]:
import shutil, os, glob

In [6]:
files = sorted(glob.glob(images_path + "/*.jpg"))
print(files)

['frames_25fps_every_5/images\\000000.jpg', 'frames_25fps_every_5/images\\000005.jpg', 'frames_25fps_every_5/images\\000010.jpg', 'frames_25fps_every_5/images\\000015.jpg', 'frames_25fps_every_5/images\\000020.jpg', 'frames_25fps_every_5/images\\000025.jpg', 'frames_25fps_every_5/images\\000030.jpg', 'frames_25fps_every_5/images\\000035.jpg', 'frames_25fps_every_5/images\\000040.jpg', 'frames_25fps_every_5/images\\000045.jpg', 'frames_25fps_every_5/images\\000050.jpg', 'frames_25fps_every_5/images\\000055.jpg', 'frames_25fps_every_5/images\\000060.jpg', 'frames_25fps_every_5/images\\000065.jpg', 'frames_25fps_every_5/images\\000070.jpg', 'frames_25fps_every_5/images\\000075.jpg', 'frames_25fps_every_5/images\\000080.jpg', 'frames_25fps_every_5/images\\000085.jpg', 'frames_25fps_every_5/images\\000090.jpg', 'frames_25fps_every_5/images\\000095.jpg', 'frames_25fps_every_5/images\\000100.jpg', 'frames_25fps_every_5/images\\000105.jpg', 'frames_25fps_every_5/images\\000110.jpg', 'frames_25

One nuance in here: The set is split by time interval

In [7]:
n = len(files)
splits = {"train": files[:int(.7*n)],
          "val":   files[int(.7*n):int(.85*n)],
          "test":  files[int(.85*n):]}

print(f"training set: {len(splits['train'])}")
print(f"validation set: {len(splits['val'])}")
print(f"test set: {len(splits['test'])}")

print(f"training set: {splits['train']}")
print(f"validation set: {splits['val']}")
print(f"test set: {splits['test']}")


training set: 188
validation set: 40
test set: 41
training set: ['frames_25fps_every_5/images\\000000.jpg', 'frames_25fps_every_5/images\\000005.jpg', 'frames_25fps_every_5/images\\000010.jpg', 'frames_25fps_every_5/images\\000015.jpg', 'frames_25fps_every_5/images\\000020.jpg', 'frames_25fps_every_5/images\\000025.jpg', 'frames_25fps_every_5/images\\000030.jpg', 'frames_25fps_every_5/images\\000035.jpg', 'frames_25fps_every_5/images\\000040.jpg', 'frames_25fps_every_5/images\\000045.jpg', 'frames_25fps_every_5/images\\000050.jpg', 'frames_25fps_every_5/images\\000055.jpg', 'frames_25fps_every_5/images\\000060.jpg', 'frames_25fps_every_5/images\\000065.jpg', 'frames_25fps_every_5/images\\000070.jpg', 'frames_25fps_every_5/images\\000075.jpg', 'frames_25fps_every_5/images\\000080.jpg', 'frames_25fps_every_5/images\\000085.jpg', 'frames_25fps_every_5/images\\000090.jpg', 'frames_25fps_every_5/images\\000095.jpg', 'frames_25fps_every_5/images\\000100.jpg', 'frames_25fps_every_5/images\\00

In [10]:
for name, fs in splits.items():
    os.makedirs(f"dataset/images/{name}", exist_ok=True)
    os.makedirs(f"dataset/labels/{name}", exist_ok=True)

    count_labels = 0

    for f in fs:
        base = os.path.splitext(os.path.basename(f))[0]
        
        # Copy the image
        shutil.copy2(f, f"dataset/images/{name}/")

        # Find the corresponding label file
        label_file = f"{labels_path}/train/{base}.txt"
        destination_label = f"dataset/labels/{name}/{base}.txt"

        if os.path.exists(label_file):
            # Label exists: copy it
            shutil.copy2(label_file, destination_label)
            count_labels += 1
        else:
            # No label exported: create an empty label file
            open(destination_label, "w").close()

    print(f"Copied {len(fs)} images and {count_labels} labels to dataset/{name}/")

Copied 188 images and 27 labels to dataset/train/
Copied 40 images and 15 labels to dataset/val/
Copied 41 images and 6 labels to dataset/test/


In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")

results = model.train(
    data="/content/dataset/data.yaml",
    epochs=100,
    imgsz=960,
    batch=-1,
    device=0,
    workers=2,
    degrees=5,
    flipud=0.0,
    fliplr=0.5,
    patience=20,
    project="/content/runs",
    name="name_tag_detection"
)

Ultralytics 8.4.177  Python-3.12.3 torch-2.5.1+cu118 CPU (Intel Core Ultra 7 155H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/data.yaml, degrees=5, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=None, opset=None, optimize=False, optimizer=auto, ove

: 

In [ ]:
from IPython.display import display, Image
from pathlib import Path

run_dir = Path("/content/runs/name_tag_detection")

# Display the training curves generated by Ultralytics
results_plot = run_dir / "results.png"

if results_plot.exists():
    display(Image(filename=str(results_plot)))
else:
    print("results.png not found. Check whether training completed.")

In [ ]:
import pandas as pd

df = pd.read_csv(run_dir / "results.csv")
display(df.tail())

In [ ]:
from ultralytics import YOLO

best_model = YOLO(
    "/content/runs/name_tag_detection/weights/best.pt"
)

test_metrics = best_model.val(
    data="/content/dataset/data.yaml",
    split="test",
    imgsz=960,
    device=0
)

print("Test evaluation complete.")

In [ ]:
from ultralytics import YOLO

best_model = YOLO(
    "/content/runs/name_tag_detection/weights/best.pt"
)

best_model.predict(
    source="/content/dataset/images/test",
    imgsz=960,
    conf=0.25,
    device=0,
    save=True,
    project="/content/runs",
    name="name_tag_predictions"
)

In [ ]:
from pathlib import Path
from IPython.display import display, Image

pred_dir = Path("/content/runs/name_tag_predictions")
pred_images = sorted(
    list(pred_dir.glob("*.jpg")) +
    list(pred_dir.glob("*.png"))
)

for path in pred_images[:5]:
    print(path.name)
    display(Image(filename=str(path)))